# Module 04 — Offer Strength Simulator (v2)

**How to use**: Edit the `── YOUR PARAMETERS ──` cell below, then `Kernel → Restart & Run All`.

### What this produces
1. Bayesian zip market profile — posterior hotness premium with uncertainty
2. Single-point offer analysis with verdict (Bayesian-calibrated P(win))
3. Bid sweep chart — win probability curve with posterior credible band
4. Bid-for-probability table — "what do I need to bid for X% odds?"
5. Full bid strategy report

### Model — two-stage Bayesian approach

**Stage 1** — LightGBM hedonic regression gives the *fair value point estimate* $\hat{y}$.

**Stage 2** — Hierarchical Bayesian model on CV residuals $r_{ij} = y_{ij} - \hat{y}_{ij}$:

$$r_{ij} \sim \mathcal{N}(\mu_j,\ \sigma_j)$$

$$\mu_j \sim \mathcal{N}(\mu_\text{pop},\ \tau_\mu) \quad \text{(zip hotness premium, partial pooling)}$$

$$\sigma_j \sim \text{HalfNormal}(\sigma_\text{pop}) \quad \text{(zip price dispersion, partial pooling)}$$

Partial pooling is critical: zip codes with fewer than ~15 sales get their hotness premium
*shrunk toward the population mean*, preventing noisy extreme estimates in thin markets.

**Win probability** using posterior means:

$$P(\text{win}) = \Phi\!\left(\frac{b - (\hat{y} + \mu_j)}{\sigma_j}\right)$$

In hot markets where list price > fair value, the clearing anchor shifts to list price.
The posterior uncertainty in $\mu_j$ and $\sigma_j$ is propagated to produce a
**80% credible interval** on $P(\text{win})$, not just a point estimate.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')

saved     = joblib.load('models/hedonic_model_filtered.joblib')
resid_std = saved['residual_std']

df = pd.read_csv('data/model_results_filtered.csv')
df['zip'] = df['zip'].astype(str).str.zfill(5)
cv_col = 'cv_predicted' if 'cv_predicted' in df.columns else 'predicted'
df['residual'] = df['PRICE'] - df[cv_col]

# ── Naive zip stats (sample means/stds from CV residuals) ─────────────────────
naive_zip_stats = (
    df.groupby('zip')
    .agg(n=('PRICE','count'), med_price=('PRICE','median'),
         hotness_prem=('residual','median'), price_std=('residual','std'),
         city=('CITY', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
    .reset_index()
)
naive_zip_stats['price_std'] = naive_zip_stats['price_std'].fillna(resid_std)

def _dom_f(dom): return max(0.5, 1.0 - (dom - 30) / 300) if dom >= 30 else 1.0

print(f'Ready. {len(df):,} sold homes | {df["zip"].nunique()} zip codes | σ = ${resid_std:,.0f}')
print(f'Model version: {saved.get("feature_version", 1)}')

Ready. 6,345 sold homes | 106 zip codes | σ = $88,512
Model version: 3


In [2]:
# ── Bayesian Hierarchical Model for Zip-Level Market Characterisation ─────────
#
# Model (non-centred parameterisation):
#   r_ij ~ Normal(mu_j, sigma_j)
#   mu_j   = mu_pop + mu_z_raw_j * sigma_mu     — non-centred (avoids funnel)
#   sigma_j = sigma_z_raw_j * sigma_pop          — non-centred
#   mu_z_raw_j    ~ Normal(0, 1)
#   sigma_z_raw_j ~ HalfNormal(1)
#   mu_pop    ~ Normal(0, 15000)
#   sigma_mu  ~ HalfNormal(20000)
#   sigma_pop ~ HalfNormal(60000)
#
# Why non-centred? When group-level variance (sigma_mu) is small relative to
# the data likelihood, the centred parameterisation creates a funnel geometry
# that causes NUTS divergences. The non-centred form moves the correlation
# into a deterministic transformation, giving the sampler a much easier surface.
# This is expected here: LightGBM with spatial features already captures most
# location signal, so residual zip-level effects are modest.
#
# MCMC: NUTS, 2 chains × 2,000 draws (1,000 tune), target_accept=0.95.
# Convergence: R-hat < 1.01 and ESS > 400 per raw parameter.
# Runtime: ~2–3 min on first run; trace cached to /tmp for reruns.
import pymc as pm
import arviz as az
import os

_CACHE = '/tmp/bayes_zip_trace_nc.joblib'   # 'nc' = non-centred

zip_counts = df['zip'].value_counts()
valid_zips  = sorted(zip_counts[zip_counts >= 5].index)
res_b = df[df['zip'].isin(valid_zips)].copy()
zip_idx_map = {z: i for i, z in enumerate(valid_zips)}
Z   = res_b['zip'].map(zip_idx_map).values.astype(int)
r   = res_b['residual'].values.astype(float)
n_z = len(valid_zips)
print(f"Bayesian model: {len(r):,} observations across {n_z} zip codes")

if os.path.exists(_CACHE):
    print("Loading cached Bayesian trace (non-centred)...")
    _cached = joblib.load(_CACHE)
    trace         = _cached['trace']
    zip_bayes_df  = _cached['zip_bayes_df']
    mu_z_samples  = _cached['mu_z_samples']
    sig_z_samples = _cached['sig_z_samples']
else:
    print("Building PyMC hierarchical residual model (non-centred)...")
    print("Sampling NUTS (2,000 draws, 1,000 tune, 2 chains, target_accept=0.95)...")
    with pm.Model() as hier_model:
        # Population hyperpriors
        mu_pop    = pm.Normal('mu_pop',    mu=0.0,    sigma=15_000)
        sigma_mu  = pm.HalfNormal('sigma_mu',  sigma=20_000)
        sigma_pop = pm.HalfNormal('sigma_pop', sigma=60_000)

        # Non-centred zip effects — raw ∼ N(0,1) / HalfNormal(1)
        # Actual effects: mu_z = mu_pop + mu_z_raw * sigma_mu
        # Decouples hyperpriors from group geometry → no funnel divergences
        mu_z_raw    = pm.Normal('mu_z_raw',    mu=0, sigma=1, shape=n_z)
        sigma_z_raw = pm.HalfNormal('sigma_z_raw', sigma=1,   shape=n_z)

        mu_z    = pm.Deterministic('mu_z',    mu_pop + mu_z_raw * sigma_mu)
        sigma_z = pm.Deterministic('sigma_z', sigma_z_raw * sigma_pop)

        # Likelihood
        _obs = pm.Normal('obs', mu=mu_z[Z], sigma=sigma_z[Z], observed=r)

        trace = pm.sample(
            2000, tune=1000, chains=2, cores=2,
            target_accept=0.95, random_seed=42, progressbar=True,
        )

    # Check convergence on the raw (non-Deterministic) parameters
    raw_vars = ['mu_pop', 'sigma_mu', 'sigma_pop', 'mu_z_raw', 'sigma_z_raw']
    rhat_raw = az.rhat(trace, var_names=raw_vars)
    all_rhats = []
    for v in raw_vars:
        all_rhats.extend(list(np.atleast_1d(rhat_raw[v].values).ravel()))
    max_r = max(float(x) for x in all_rhats if not np.isnan(x))
    n_div  = int(trace.sample_stats['diverging'].values.sum())
    print(f"\nMax R-hat (raw params): {max_r:.4f}  "
          f"({'OK' if max_r < 1.01 else 'WARNING: consider more tuning steps'})")
    print(f"Divergences: {n_div}  ({'OK' if n_div == 0 else 'WARNING: try target_accept=0.98'})")

    # Posterior summaries per zip
    mu_z_post  = trace.posterior['mu_z'].values    # (chains, draws, n_z)
    sig_z_post = trace.posterior['sigma_z'].values
    mu_z_flat  = mu_z_post.reshape(-1, n_z)
    sig_z_flat = sig_z_post.reshape(-1, n_z)

    zip_bayes_df = pd.DataFrame({
        'zip':          valid_zips,
        'n_sales':      [int(zip_counts[z]) for z in valid_zips],
        'mu_z_mean':    mu_z_flat.mean(axis=0),
        'mu_z_lo80':    np.percentile(mu_z_flat, 10, axis=0),
        'mu_z_hi80':    np.percentile(mu_z_flat, 90, axis=0),
        'sigma_z_mean': sig_z_flat.mean(axis=0),
        'sigma_z_lo80': np.percentile(sig_z_flat, 10, axis=0),
        'sigma_z_hi80': np.percentile(sig_z_flat, 90, axis=0),
    }).merge(naive_zip_stats[['zip','city','hotness_prem','price_std']],
             on='zip', how='left')
    zip_bayes_df['shrinkage_pct'] = (
        (1 - zip_bayes_df['mu_z_mean'] / zip_bayes_df['hotness_prem'].replace(0, np.nan))
        .abs().round(2)
    )

    mu_z_samples  = mu_z_flat
    sig_z_samples = sig_z_flat
    joblib.dump({'trace': trace, 'zip_bayes_df': zip_bayes_df,
                 'mu_z_samples': mu_z_flat, 'sig_z_samples': sig_z_flat}, _CACHE)
    print(f"Trace cached → {_CACHE}")

# ── Population-level posterior summary ───────────────────────────────────────
print(f"\n{'='*64}")
print("BAYESIAN POSTERIOR — POPULATION-LEVEL PARAMETERS")
print(f"{'='*64}")
pop_summ = az.summary(trace, var_names=['mu_pop','sigma_mu','sigma_pop'],
                      hdi_prob=0.80, round_to=0)
print(pop_summ[['mean','sd','hdi_10%','hdi_90%','r_hat']].to_string())

print(f"\n{'='*64}")
print("TOP 10 HOT ZIPS — Bayesian posterior mu_z (partial pooling)")
print(f"{'='*64}")
top_hot = zip_bayes_df.sort_values('mu_z_mean', ascending=False).head(10)
for _, row in top_hot.iterrows():
    city = row.get('city','') or ''
    print(f"  {row['zip']} {city:<18}  mu_z={row['mu_z_mean']:>+7,.0f}  "
          f"80%HDI [{row['mu_z_lo80']:>+7,.0f}, {row['mu_z_hi80']:>+7,.0f}]  "
          f"sigma={row['sigma_z_mean']:>6,.0f}  n={int(row['n_sales'])}")

thin = zip_bayes_df[zip_bayes_df['n_sales'] < 20].copy()
if not thin.empty:
    print(f"\n  Thin markets (n<20) — partial pooling shrinks naive estimates:")
    for _, row in thin.sort_values('n_sales').head(6).iterrows():
        naive = row.get('hotness_prem', 0) or 0
        print(f"    {row['zip']}  n={int(row['n_sales'])}  "
              f"naive=${naive:>+7,.0f}  bayes=${row['mu_z_mean']:>+7,.0f}")

# ── Helper: look up Bayesian zip stats ────────────────────────────────────────
def _zstats_bayes(z):
    """Return Bayesian posterior summary for a zip code."""
    z = str(z).zfill(5)
    row = zip_bayes_df[zip_bayes_df['zip'] == z]
    if not row.empty:
        rv = row.iloc[0]
        idx = valid_zips.index(z) if z in valid_zips else None
        return dict(zip=z,
                    city=rv.get('city','') or '',
                    n=int(rv['n_sales']),
                    hotness_prem=float(rv['mu_z_mean']),
                    hotness_prem_lo=float(rv['mu_z_lo80']),
                    hotness_prem_hi=float(rv['mu_z_hi80']),
                    price_std=float(rv['sigma_z_mean']),
                    price_std_lo=float(rv['sigma_z_lo80']),
                    price_std_hi=float(rv['sigma_z_hi80']),
                    naive_prem=float(rv['hotness_prem']) if 'hotness_prem' in rv else 0.0,
                    mu_z_idx=idx,
                    mu_z_samples=mu_z_samples[:, idx] if idx is not None else None,
                    sig_z_samples=sig_z_samples[:, idx] if idx is not None else None)
    # Unknown zip — fall back to population posterior
    pop_post = trace.posterior['mu_pop'].values.ravel()
    sig_post = trace.posterior['sigma_pop'].values.ravel()
    return dict(zip=z, city='', n=0,
                hotness_prem=float(pop_post.mean()),
                hotness_prem_lo=float(np.percentile(pop_post, 10)),
                hotness_prem_hi=float(np.percentile(pop_post, 90)),
                price_std=float(sig_post.mean()),
                price_std_lo=float(np.percentile(sig_post, 10)),
                price_std_hi=float(np.percentile(sig_post, 90)),
                naive_prem=0.0, mu_z_idx=None,
                mu_z_samples=None, sig_z_samples=None)

Bayesian model: 6,290 observations across 81 zip codes
Loading cached Bayesian trace (non-centred)...

BAYESIAN POSTERIOR — POPULATION-LEVEL PARAMETERS
              mean      sd  hdi_10%  hdi_90%  r_hat
mu_pop      -201.0   824.0  -1235.0    836.0    1.0
sigma_mu    1168.0   866.0      1.0   1885.0    1.0
sigma_pop  66729.0  5512.0  58604.0  72648.0    1.0

TOP 10 HOT ZIPS — Bayesian posterior mu_z (partial pooling)
  19078 Ridley Park         mu_z=   +330  80%HDI [ -1,432,  +2,405]  sigma=44,795  n=49
  19044 Horsham             mu_z=   +135  80%HDI [ -1,702,  +2,068]  sigma=46,995  n=45
  19464 Pottstown           mu_z=   +120  80%HDI [ -1,508,  +1,948]  sigma=48,780  n=254
  18076 Red Hill            mu_z=   +119  80%HDI [ -1,662,  +2,074]  sigma=41,635  n=14
  19029 Essington           mu_z=    +99  80%HDI [ -1,739,  +2,155]  sigma=33,717  n=5
  19001 Abington            mu_z=    +98  80%HDI [ -1,677,  +2,055]  sigma=57,601  n=142
  19043 Holmes              mu_z=    +35  80%HDI [

In [3]:
# ── YOUR PARAMETERS — edit this cell, then Kernel → Restart & Run All ────────

ZIP_CODE   = '19083'     # 5-digit zip code
FAIR_VALUE = 635_000     # Model fair-value estimate (from dashboard or 03_evaluation.ipynb)
LIST_PRICE = 670_000     # Seller's asking price (set equal to FAIR_VALUE if unknown)
YOUR_BID   = 640_000     # Your planned offer

DOM        = 2          # Current days on market
MAX_BUDGET = 750_000    # Your hard ceiling

# Escalation clause — set ESC_BASE = None to skip
ESC_BASE      = 640_000  # Opening escalation bid
ESC_INCREMENT = 5_000   # Beat any competing offer by this much
ESC_CAP       = 675_000  # Maximum you will pay

In [4]:
# ── Run analysis ─────────────────────────────────────────────────────────────
zs = _zstats_bayes(ZIP_CODE)

# ── Clearing price model ──────────────────────────────────────────────────────
# In hot markets (mu_z > 0 AND list_price > fair_value), sellers anchor to ask.
# Bidding below list against multiple competing offers near-guarantees a loss.
hot_market = (zs['hotness_prem'] > 0) and (LIST_PRICE > FAIR_VALUE)
list_val   = float(LIST_PRICE)
base_price = list_val if hot_market else FAIR_VALUE
gap_adj    = 0.0 if hot_market else (FAIR_VALUE - list_val) * 0.25
dom_factor = _dom_f(DOM)

# Point estimate clearing price
mu_clear = base_price + zs['hotness_prem'] * dom_factor + gap_adj
sig      = zs['price_std']
pw_point = float(stats.norm.cdf(YOUR_BID, mu_clear, sig))

# ── Bayesian credible interval on P(win) ─────────────────────────────────────
# Use posterior samples stored in zs (pre-sliced for this zip by _zstats_bayes).
# Falls back to population posterior for unknown zips.
if zs['mu_z_samples'] is not None:
    _mu_z_post  = zs['mu_z_samples']    # (n_samples,)
    _sig_z_post = zs['sig_z_samples']
else:
    _mu_z_post  = trace.posterior['mu_pop'].values.ravel()
    _sig_z_post = trace.posterior['sigma_pop'].values.ravel()

_mu_clear_post = base_price + _mu_z_post * dom_factor + gap_adj
_pw_post       = stats.norm.cdf(YOUR_BID, _mu_clear_post, np.abs(_sig_z_post) + 1e-6)
pw_lo80, pw_hi80 = np.percentile(_pw_post, [10, 90])
pw_median        = float(np.median(_pw_post))

# Verdict from point estimate
if   pw_point >= 0.80: verdict = "VERY STRONG  — likely the top bid"
elif pw_point >= 0.65: verdict = "STRONG       — solid chance of win"
elif pw_point >= 0.50: verdict = "COMPETITIVE  — near expected clearing"
elif pw_point >= 0.35: verdict = "MODERATE     — may lose to higher bids"
else:                  verdict = "WEAK         — well below clearing"

# ── 1. Zip market profile ─────────────────────────────────────────────────────
print(f'\n{"="*62}')
print(f'  ZIP {ZIP_CODE} — {zs["city"]}  (n={zs["n"]} comparable sales)')
print(f'{"="*62}')
print(f'  Bayesian hotness premium  (posterior mean):    ${zs["hotness_prem"]:>+8,.0f}')
print(f'  80% HDI for hotness prem:                  [${zs["hotness_prem_lo"]:>+8,.0f}, ${zs["hotness_prem_hi"]:>+8,.0f}]')
print(f'  Bayesian price std σ      (posterior mean):    ${zs["price_std"]:>8,.0f}')
print(f'  80% HDI for price std:                     [${zs["price_std_lo"]:>8,.0f}, ${zs["price_std_hi"]:>8,.0f}]')
if abs(zs.get('naive_prem', 0) or 0) > 100:
    print(f'  Naive sample premium (pre-pooling):            ${zs["naive_prem"]:>+8,.0f}')
    print(f'  Partial pooling effect:        naive → Bayesian (shrinkage toward population)')
if DOM >= 30:
    print(f'  DOM cooldown factor ({DOM}d):                  {dom_factor:>8.2f}×')
print(f'  Clearing price anchor:     {"list price (hot market)" if hot_market else "fair value (neutral/cool)"}')

# ── 2. Offer analysis ─────────────────────────────────────────────────────────
print(f'\n{"-"*62}')
print(f'  OFFER ANALYSIS')
print(f'{"-"*62}')
print(f'  List price           : ${LIST_PRICE:>12,.0f}  (FV is {(FAIR_VALUE-LIST_PRICE)/LIST_PRICE*100:+.1f}% vs list)')
print(f'  Fair value (model)   : ${FAIR_VALUE:>12,.0f}')
print(f'  Expected clear price : ${mu_clear:>12,.0f}')
print(f'  Your bid             : ${YOUR_BID:>12,.0f}  ({(YOUR_BID-FAIR_VALUE)/FAIR_VALUE*100:+.1f}% vs fair value)')
print(f'')
print(f'  P(win) — point est.         : {pw_point*100:>10.1f}%')
print(f'  P(win) — posterior median   : {pw_median*100:>10.1f}%')
print(f'  P(win) — 80% credible interval: [{pw_lo80*100:.1f}%, {pw_hi80*100:.1f}%]')
print(f'  Verdict              : {verdict}')

# ── 3. Bid sweep chart with posterior uncertainty band ────────────────────────
lo_b = FAIR_VALUE * 0.82
hi_b = min(FAIR_VALUE * 1.30, MAX_BUDGET * 1.06)
bids = np.linspace(lo_b, hi_b, 400)

# Point estimate curve
probs_pt = stats.norm.cdf(bids, mu_clear, sig) * 100

# Posterior uncertainty band: 500 samples propagated through the CDF
n_samp = min(500, len(_mu_z_post))
rng    = np.random.default_rng(42)
idx_s  = rng.choice(len(_mu_z_post), size=n_samp, replace=False)
mu_s   = (base_price + _mu_z_post[idx_s] * dom_factor + gap_adj)[:, None]  # (n_samp, 1)
sig_s  = (np.abs(_sig_z_post[idx_s]) + 1e-6)[:, None]
probs_mat  = stats.norm.cdf(bids[None, :], mu_s, sig_s) * 100              # (n_samp, n_bids)
p_lo_band  = np.percentile(probs_mat, 10, axis=0)
p_hi_band  = np.percentile(probs_mat, 90, axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.concatenate([bids, bids[::-1]]),
                          y=np.concatenate([p_hi_band, p_lo_band[::-1]]),
                          fill='toself', fillcolor='rgba(41,128,185,0.12)',
                          line=dict(color='rgba(255,255,255,0)'),
                          name='80% credible band', showlegend=True))
fig.add_trace(go.Scatter(x=bids, y=probs_pt, mode='lines',
                          line=dict(color='#2980b9', width=2.5),
                          name='P(win) — posterior mean',
                          hovertemplate='Bid: $%{x:,.0f}<br>P(win): %{y:.1f}%<extra></extra>'))
for lo_p, hi_p, col, lbl in [(65,100,'rgba(39,174,96,0.08)','Strong (65%+)'),
                               (40,65, 'rgba(243,156,18,0.08)','Competitive (40–65%)'),
                               (0,  40,'rgba(231,76,60,0.08)', 'Weak (<40%)')]:
    fig.add_hrect(y0=lo_p, y1=hi_p, fillcolor=col, line_width=0,
                  annotation_text=lbl, annotation_position='right')
fig.add_vline(x=YOUR_BID,   line_dash='dash',  line_color='#e74c3c', line_width=1.5,
              annotation_text=f'Your bid ${YOUR_BID:,.0f} ({pw_point*100:.0f}%)',
              annotation_position='top right')
fig.add_vline(x=FAIR_VALUE, line_dash='dot',   line_color='#7f8c8d', line_width=1,
              annotation_text=f'Fair value ${FAIR_VALUE:,.0f}', annotation_position='top left')
fig.add_vline(x=mu_clear,   line_dash='solid', line_color='#27ae60', line_width=1,
              annotation_text=f'E[clear] ${mu_clear:,.0f}', annotation_position='bottom right')
if lo_b <= MAX_BUDGET <= hi_b:
    fig.add_vline(x=MAX_BUDGET, line_dash='dot', line_color='#8e44ad', line_width=1,
                  annotation_text=f'Max ${MAX_BUDGET:,.0f}', annotation_position='top left')
fig.update_layout(
    title=f'Win Probability vs Bid — ZIP {ZIP_CODE} ({zs["city"]})<br>'
          f'<sup>Shaded band = 80% posterior credible interval from Bayesian model</sup>',
    xaxis_title='Offer Price', yaxis_title='P(win) %',
    xaxis_tickformat='$,.0f', yaxis=dict(range=[0,105], ticksuffix='%'),
    height=440, plot_bgcolor='white', paper_bgcolor='white',
)
fig.show()

# ── 4. Bid-for-probability table ─────────────────────────────────────────────
print(f'\n{"-"*62}')
print(f'  BIDS NEEDED FOR TARGET WIN PROBABILITY')
print(f'  (Bayesian posterior mean clearing price = ${mu_clear:,.0f})')
print(f'{"-"*62}')
print(f'  {"Target":>8}  {"Required Bid":>14}  {"vs Fair Value":>13}  {"vs List":>8}  Notes')
print(f'  {"-"*60}')
for p in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    b    = float(stats.norm.ppf(p, mu_clear, sig))
    dpct = (b - FAIR_VALUE) / FAIR_VALUE * 100
    lvs  = (b - LIST_PRICE) / LIST_PRICE * 100
    note = 'OVER BUDGET' if b > MAX_BUDGET else ('◄ YOUR BID' if abs(b - YOUR_BID) < 7000 else '')
    print(f'  {p*100:>7.0f}%  ${b:>13,.0f}  {dpct:>+12.1f}%  {lvs:>+7.1f}%  {note}')

# ── 5. Bid strategy report ────────────────────────────────────────────────────
print(f'\n{"━"*62}')
print(f'  BID STRATEGY REPORT — BAYESIAN CALIBRATION')
print(f'  ZIP {ZIP_CODE} ({zs["city"]})  |  List ${LIST_PRICE:,.0f}  |  FV ${FAIR_VALUE:,.0f}  |  DOM {DOM}')
print(f'{"━"*62}')
print(f'  {"Bid":>14}  {"vs Fair Val":>12}  {"P(win)":>8}  {"80% CI":>18}  Tier')
print(f'  {"-"*62}')
for p_t in [0.30, 0.45, 0.55, 0.65, 0.75, 0.85]:
    b    = float(stats.norm.ppf(p_t, mu_clear, sig))
    dpct = (b - FAIR_VALUE) / FAIR_VALUE * 100
    _ci  = stats.norm.cdf(b, _mu_clear_post, np.abs(_sig_z_post) + 1e-6)
    ci_lo, ci_hi = np.percentile(_ci, [10, 90])
    tier = 'OVER BUDGET' if b > MAX_BUDGET else (
        'Strong' if p_t >= 0.75 else ('Competitive' if p_t >= 0.55 else 'Speculative'))
    you  = '  ◄ your bid' if abs(b - YOUR_BID) < 8000 else ''
    print(f'  ${b:>13,.0f}  {dpct:>+10.1f}%  {p_t*100:>7.0f}%  '
          f'[{ci_lo*100:.0f}%, {ci_hi*100:.0f}%]  {tier}{you}')
print(f'  {"-"*62}')
_max_ci = stats.norm.cdf(MAX_BUDGET, _mu_clear_post, np.abs(_sig_z_post) + 1e-6)
max_pw  = float(stats.norm.cdf(MAX_BUDGET, mu_clear, sig))
print(f'  Max budget ${MAX_BUDGET:,.0f}  →  P(win) = {max_pw*100:.1f}%  '
      f'(80% CI: [{np.percentile(_max_ci,10)*100:.0f}%, {np.percentile(_max_ci,90)*100:.0f}%])')
print(f'{"━"*62}')


  ZIP 19083 — Havertown  (n=198 comparable sales)
  Bayesian hotness premium  (posterior mean):    $     +15
  80% HDI for hotness prem:                  [$  -1,780, $  +1,878]
  Bayesian price std σ      (posterior mean):    $  73,677
  80% HDI for price std:                     [$  69,087, $  78,481]
  Naive sample premium (pre-pooling):            $  -5,316
  Partial pooling effect:        naive → Bayesian (shrinkage toward population)
  Clearing price anchor:     list price (hot market)

--------------------------------------------------------------
  OFFER ANALYSIS
--------------------------------------------------------------
  List price           : $     670,000  (FV is -5.2% vs list)
  Fair value (model)   : $     635,000
  Expected clear price : $     670,015
  Your bid             : $     640,000  (+0.8% vs fair value)

  P(win) — point est.         :       34.2%
  P(win) — posterior median   :       30.1%
  P(win) — 80% credible interval: [28.1%, 32.0%]
  Verdict          


--------------------------------------------------------------
  BIDS NEEDED FOR TARGET WIN PROBABILITY
  (Bayesian posterior mean clearing price = $670,015)
--------------------------------------------------------------
    Target    Required Bid  vs Fair Value   vs List  Notes
  ------------------------------------------------------------
       30%  $      631,379          -0.6%     -5.8%  
       40%  $      651,349          +2.6%     -2.8%  
       50%  $      670,015          +5.5%     +0.0%  
       60%  $      688,681          +8.5%     +2.8%  
       70%  $      708,651         +11.6%     +5.8%  
       80%  $      732,023         +15.3%     +9.3%  
       90%  $      764,436         +20.4%    +14.1%  OVER BUDGET

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BID STRATEGY REPORT — BAYESIAN CALIBRATION
  ZIP 19083 (Havertown)  |  List $670,000  |  FV $635,000  |  DOM 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
             Bid   vs Fair 